This joining is and verification in all done in full data and not deduped. refer to py file #5 for the same thing but on deduped data.

DATA ANALYSIS Tier 2 calls and equity questions

In [51]:
import pandas as pd

# 1. Load the Call Data (Parquet)
calls_path = '../data_processed/calls_2025_tiered_final_Re.parquet' #this is already deduped
df_calls = pd.read_parquet(calls_path)

# 2. Load the Census/RSE Mapping Data (CSV)
census_path = '../Data/calldata_census_rse_neighbourhood_joined.csv'
df_census = pd.read_csv(census_path)

# 3. Load the Census/RSE Mapping Data (CSV)
map_path = '../Data/CallData_CRA_map.csv'
df_map = pd.read_csv(map_path)

# 3. Clean and Normalize Join Keys
df_calls['dispatch_neighborhood'] = df_calls['dispatch_neighborhood'].str.strip().str.upper()
df_census['Call Data Neighbourhood'] = df_census['Call Data Neighbourhood'].str.strip().str.upper()

df_map['Call Data Neighbourhood'] = df_map['Call Data Neighbourhood'].str.strip().str.upper()



In [52]:

# --- 0) Pre-join baselines ---
n_before = len(df_calls)
n_events_before = df_calls["cad_event_number"].nunique(dropna=False) if "cad_event_number" in df_calls.columns else None

print("=== PRE-JOIN ===")
print(f"Rows (df_calls): {n_before:,}")
if n_events_before is not None:
    print(f"Unique cad_event_number: {n_events_before:,}  |  rows-per-event: {n_before/n_events_before:.3f}")

print("\ncare_tier distribution (rows):")
care_pre = (df_calls["care_tier"].astype(str).str.strip()
            .value_counts(dropna=False)
            .to_frame("count"))
care_pre["pct"] = (care_pre["count"] / care_pre["count"].sum() * 100).round(2)
print(care_pre)

# --- 1) Check right table uniqueness on join key (this is the usual blow-up cause) ---
print("\n=== RIGHT TABLE KEY CHECK (df_map) ===")
dup_keys = df_map["Call Data Neighbourhood"].astype(str).value_counts()
n_dup_keys = (dup_keys > 1).sum()
print(f"Distinct keys in df_map: {df_map['Call Data Neighbourhood'].nunique(dropna=False):,}")
print(f"Keys that appear >1 time (would cause blow-up): {n_dup_keys:,}")

if n_dup_keys > 0:
    print("\nTop duplicate join keys:")
    print(dup_keys[dup_keys > 1].head(20))

    # optional: inspect the actual duplicate rows
    # key_to_inspect = dup_keys[dup_keys > 1].index[0]
    # print(df_map[df_map["Call Data Neighbourhood"].astype(str) == str(key_to_inspect)]
    #       [["Call Data Neighbourhood", "Community Reporting Area Name"]])


=== PRE-JOIN ===
Rows (df_calls): 324,386
Unique cad_event_number: 324,386  |  rows-per-event: 1.000

care_tier distribution (rows):
                           count    pct
care_tier                              
Tier 0 - Traditional SPD  177837  54.82
Tier 1 - Potential CARE   125682  38.74
Tier 2 - Clearly CARE      20867   6.43

=== RIGHT TABLE KEY CHECK (df_map) ===
Distinct keys in df_map: 63
Keys that appear >1 time (would cause blow-up): 0


In [53]:

# --- 2) Perform join WITH validate so it errors instead of silently multiplying ---
print("\n=== JOIN ===")
df_map_small = df_map[["Call Data Neighbourhood", "Community Reporting Area Name"]].copy()

try:
    df_call_map_joined = df_calls.merge(
        df_map_small,
        left_on="dispatch_neighborhood",
        right_on="Call Data Neighbourhood",
        how="left",
        indicator=True,
        validate="m:1"  # many calls -> one map row; will throw if df_map has dup keys
    ).drop(columns=["Call Data Neighbourhood"])
    print("Join validate=m:1: PASSED (no row multiplication expected).")
except Exception as e:
    print("Join validate=m:1: FAILED (likely duplicate keys in df_map).")
    print("Error:", e)

    # Fall back to the join anyway (so you can see row counts), but *know it's not safe*
    df_call_map_joined = df_calls.merge(
        df_map_small,
        left_on="dispatch_neighborhood",
        right_on="Call Data Neighbourhood",
        how="left",
        indicator=True
    ).drop(columns=["Call Data Neighbourhood"])



=== JOIN ===
Join validate=m:1: PASSED (no row multiplication expected).


In [54]:

# --- 3) Post-join counts ---
n_after = len(df_call_map_joined)
n_events_after = df_call_map_joined["cad_event_number"].nunique(dropna=False) if "cad_event_number" in df_call_map_joined.columns else None

print("\n=== POST-JOIN ===")
print(f"Rows (df_call_map_joined): {n_after:,}  |  delta: {n_after - n_before:+,}")
if n_events_after is not None and n_events_before is not None:
    print(f"Unique cad_event_number: {n_events_after:,}  |  delta: {n_events_after - n_events_before:+,}")
    print(f"Rows-per-event after: {n_after/n_events_after:.3f}")

# join indicator breakdown
print("\n_merge breakdown:")
print(df_call_map_joined["_merge"].value_counts(dropna=False))

# care_tier distribution after join (should match pre-join if no row multiplication)
print("\ncare_tier distribution AFTER join (rows):")
care_post = (df_call_map_joined["care_tier"].astype(str).str.strip()
             .value_counts(dropna=False)
             .to_frame("count"))
care_post["pct"] = (care_post["count"] / care_post["count"].sum() * 100).round(2)
print(care_post)





=== POST-JOIN ===
Rows (df_call_map_joined): 324,386  |  delta: +0
Unique cad_event_number: 324,386  |  delta: +0
Rows-per-event after: 1.000

_merge breakdown:
_merge
both          324386
left_only          0
right_only         0
Name: count, dtype: int64

care_tier distribution AFTER join (rows):
                           count    pct
care_tier                              
Tier 0 - Traditional SPD  177837  54.82
Tier 1 - Potential CARE   125682  38.74
Tier 2 - Clearly CARE      20867   6.43


In [55]:
# 1. Isolate the unmatched records
df_unmatched = df_call_map_joined[df_call_map_joined['_merge'] == 'left_only']

# 2. Identify the unique neighborhood names that failed to match
# We want to see the count to know how much data is being lost per name
unmatched_summary = (
    df_unmatched['dispatch_neighborhood']
    .value_counts(dropna=False)
    .rename_axis('unmatched_neighborhood')
    .reset_index(name='call_count')
)

print("--- Summary of Unmatched Neighborhoods ---")
if unmatched_summary.empty:
    print("Perfect match! No records were left behind.")
else:
    print(unmatched_summary)


--- Summary of Unmatched Neighborhoods ---
Perfect match! No records were left behind.


First join is good all checks done. df_call_map_joined --> good.

Print Schema to check

In [56]:
def print_schema(df, name="df"):
    print(f"Schema for {name}  |  rows={len(df):,} cols={df.shape[1]:,}")
    for col, dt in df.dtypes.items():
        print(f"  - {col}: {dt}")

print_schema(df_call_map_joined, "df_call_map_joined")


Schema for df_call_map_joined  |  rows=324,386 cols=49
  - cad_event_number: int64
  - cad_event_clearance_description: object
  - call_type: object
  - priority: int64
  - initial_call_type: object
  - final_call_type: object
  - cad_event_original_time_queued: datetime64[us]
  - cad_event_arrived_time: datetime64[us]
  - dispatch_precinct: object
  - dispatch_sector: object
  - dispatch_beat: object
  - dispatch_longitude: object
  - dispatch_latitude: object
  - dispatch_reporting_area: object
  - cad_event_response_category: object
  - call_sign_dispatch_id: object
  - call_sign_dispatch_time: datetime64[us]
  - first_care_call_sign_at_scene_time: datetime64[us]
  - first_care_call_sign_dispatch_time: datetime64[us]
  - first_co_response_call_sign_at_scene_time: datetime64[us]
  - first_co_response_call_sign_dispatch_time: datetime64[us]
  - first_spd_call_sign_at_scene_time: datetime64[us]
  - first_spd_call_sign_dispatch_time: datetime64[us]
  - last_care_call_sign_in_service_tim

In [57]:
def print_schema(df, name="df"):
    print(f"Schema for {name}  |  rows={len(df):,} cols={df.shape[1]:,}")
    for col, dt in df.dtypes.items():
        print(f"  - {col}: {dt}")

print_schema(df_census, "df_census")

Schema for df_census  |  rows=179 cols=48
  - OBJECTID: int64
  - GEOID: int64
  - Community Reporting Area Name: object
  - Community Reporting Area Neighborhoods: object
  - Call Data Neighbourhood: object
  - Share of population who are people of color: float64
  - Share of population who speak English less than very well: float64
  - Share of population who are foreign born: float64
  - Share of population with income below 200% of poverty level: float64
  - Share of population 25 and older with less than a bachelor's degree: float64
  - Share of adults with no leisure-time physical activity: float64
  - Share of adults with diagnosed diabetes: float64
  - Share of adults with obesity: float64
  - Share of adults reporting mental health is not good: float64
  - Share of adults with asthma: float64
  - Life expectancy at birth: float64
  - Share of adults with disability: float64
  - Percentile people of color: float64
  - Percentile speak English less than very well: float64
  - Pe

In [63]:
import numpy as np

lat_col = "dispatch_latitude"
lon_col = "dispatch_longitude"

def coord_quality_report(df, col):
    s_raw = df[col]
    s_str = s_raw.astype("string")

    is_null = s_raw.isna()
    is_redacted = s_str.str.strip().str.upper().eq("REDACTED")

    s_num = pd.to_numeric(s_raw, errors="coerce")
    is_non_numeric = s_num.isna() & (~is_null)  # includes REDACTED, blanks, etc.

    out = {
        "n_rows": len(df),
        "null_count": int(is_null.sum()),
        "null_pct": float(is_null.mean()),
        "redacted_count": int(is_redacted.sum()),
        "redacted_pct": float(is_redacted.mean()),
        "non_numeric_count": int(is_non_numeric.sum()),
        "non_numeric_pct": float(is_non_numeric.mean()),
        "numeric_count": int((~s_num.isna()).sum()),
        "numeric_pct": float((~s_num.isna()).mean()),
    }

    # Top non-numeric raw values (helpful to see what's going on)
    top_bad = (s_str[is_non_numeric]
               .str.strip()
               .value_counts()
               .head(10))

    return out, top_bad

lat_stats, lat_top_bad = coord_quality_report(df_calls, lat_col)
lon_stats, lon_top_bad = coord_quality_report(df_calls, lon_col)

print("=== LATITUDE QUALITY ===")
for k, v in lat_stats.items():
    if k.endswith("_pct"):
        print(f"{k}: {v:.2%}")
    else:
        print(f"{k}: {v:,}")
print("\nTop non-numeric latitude values:")
print(lat_top_bad)

print("\n=== LONGITUDE QUALITY ===")
for k, v in lon_stats.items():
    if k.endswith("_pct"):
        print(f"{k}: {v:.2%}")
    else:
        print(f"{k}: {v:,}")
print("\nTop non-numeric longitude values:")
print(lon_top_bad)

# Rows with BOTH coords usable numerically
lat_num = pd.to_numeric(df_calls[lat_col], errors="coerce")
lon_num = pd.to_numeric(df_calls[lon_col], errors="coerce")

both_numeric = lat_num.notna() & lon_num.notna()
both_redacted = (df_calls[lat_col].astype("string").str.strip().str.upper().eq("REDACTED") &
                 df_calls[lon_col].astype("string").str.strip().str.upper().eq("REDACTED"))

print("\n=== BOTH COORDS ===")
print(f"Both numeric:   {both_numeric.sum():,} ({both_numeric.mean():.2%})")
print(f"Both REDACTED:  {both_redacted.sum():,} ({both_redacted.mean():.2%})")
print(f"Either non-numeric: {(~both_numeric).sum():,} ({(~both_numeric).mean():.2%})")


=== LATITUDE QUALITY ===
n_rows: 324,386
null_count: 0
null_pct: 0.00%
redacted_count: 74,144
redacted_pct: 22.86%
non_numeric_count: 74,144
non_numeric_pct: 22.86%
numeric_count: 250,242
numeric_pct: 77.14%

Top non-numeric latitude values:
dispatch_latitude
REDACTED    74144
Name: count, dtype: Int64

=== LONGITUDE QUALITY ===
n_rows: 324,386
null_count: 0
null_pct: 0.00%
redacted_count: 74,144
redacted_pct: 22.86%
non_numeric_count: 74,144
non_numeric_pct: 22.86%
numeric_count: 250,242
numeric_pct: 77.14%

Top non-numeric longitude values:
dispatch_longitude
REDACTED    74144
Name: count, dtype: Int64

=== BOTH COORDS ===
Both numeric:   250,242 (77.14%)
Both REDACTED:  74,144 (22.86%)
Either non-numeric: 74,144 (22.86%)


Given 23% of the deduped calls have redacted lat/long, lets see if they belong a particular neighbourhood.

In [65]:

from pathlib import Path
df_calls_redactedlatlong = df_calls

lat_col = "dispatch_latitude"
lon_col = "dispatch_longitude"
nbhd_col = "Community Reporting Area Name" if "Community Reporting Area Name" in df_calls_redactedlatlong.columns else "dispatch_neighborhood"

# 1) Filter to REDACTED coords (both lat and lon)
lat_red = df_calls_redactedlatlong[lat_col].astype("string").str.strip().str.upper().eq("REDACTED")
lon_red = df_calls_redactedlatlong[lon_col].astype("string").str.strip().str.upper().eq("REDACTED")

df_redacted = df_calls_redactedlatlong[lat_red & lon_red].copy()

print(f"Redacted rows: {len(df_redacted):,} ({len(df_redacted)/len(df):.2%} of all rows)")

# 2) Neighborhood distribution among redacted rows
nbhd_dist = (df_redacted[nbhd_col]
             .fillna("<<NA>>")
             .value_counts(dropna=False)
             .to_frame("n_redacted"))

nbhd_dist["pct_of_redacted"] = (nbhd_dist["n_redacted"] / nbhd_dist["n_redacted"].sum() * 100).round(2)

print("\nTop 20 neighborhoods by # redacted rows:")
print(nbhd_dist.head(20))

# Save to CSV
out_path = Path("../data_processed/redacted_neighborhood_distribution_indeduped_callsdf.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
nbhd_dist.to_csv(out_path, index=False)


Redacted rows: 74,144 (5.42% of all rows)

Top 20 neighborhoods by # redacted rows:
                                  n_redacted  pct_of_redacted
dispatch_neighborhood                                        
UNKNOWN                                 5643             7.61
CHINATOWN/INTERNATIONAL DISTRICT        4493             6.06
CAPITOL HILL                            4436             5.98
DOWNTOWN COMMERCIAL                     4273             5.76
-                                       4251             5.73
NORTHGATE                               3354             4.52
SLU/CASCADE                             2440             3.29
ROXHILL/WESTWOOD/ARBOR HEIGHTS          2170             2.93
FIRST HILL                              2061             2.78
QUEEN ANNE                              1890             2.55
BALLARD SOUTH                           1840             2.48
UNIVERSITY                              1704             2.30
LAKECITY                                1664    

In [66]:
import pandas as pd
import numpy as np

# --- Load raw parquet ---
raw_path = "../data_processed/calls_2025_fresh_api.parquet"
raw = pd.read_parquet(raw_path)

# --- Dedup df (already in memory). Change if needed ---
dedup = df_calls

id_col = "cad_event_number"
lat_col = "dispatch_latitude"
lon_col = "dispatch_longitude"

def redacted_mask(df):
    lat_red = df[lat_col].astype("string").str.strip().str.upper().eq("REDACTED")
    lon_red = df[lon_col].astype("string").str.strip().str.upper().eq("REDACTED")
    return lat_red & lon_red

def numeric_mask(df):
    lat_num = pd.to_numeric(df[lat_col], errors="coerce")
    lon_num = pd.to_numeric(df[lon_col], errors="coerce")
    return lat_num.notna() & lon_num.notna()

print("=== BASIC SHAPES ===")
print("Raw rows:", f"{len(raw):,}", "| Raw unique incidents:", f"{raw[id_col].nunique(dropna=False):,}")
print("Dedup rows:", f"{len(dedup):,}", "| Dedup unique incidents:", f"{dedup[id_col].nunique(dropna=False):,}")

print("\n=== REDACTED % (ROW-LEVEL) ===")
print("Raw REDACTED %:", f"{redacted_mask(raw).mean():.2%}")
print("Dedup REDACTED %:", f"{redacted_mask(dedup).mean():.2%}")

print("\n=== NUMERIC COORDS % (ROW-LEVEL) ===")
print("Raw both-numeric %:", f"{numeric_mask(raw).mean():.2%}")
print("Dedup both-numeric %:", f"{numeric_mask(dedup).mean():.2%}")

# --- Incident-level in RAW: do incidents have mixed coord states? ---
raw_flags = raw[[id_col, lat_col, lon_col]].copy()
raw_flags["is_redacted"] = redacted_mask(raw).values
raw_flags["has_numeric"] = numeric_mask(raw).values

event_mix = raw_flags.groupby(id_col).agg(
    n_rows=(id_col, "size"),
    any_redacted=("is_redacted", "any"),
    any_numeric=("has_numeric", "any"),
)
event_mix["mixed_numeric_and_redacted"] = event_mix["any_redacted"] & event_mix["any_numeric"]

print("\n=== RAW INCIDENT-LEVEL COORD PATTERNS ===")
print("Incidents with ANY numeric coords:", f"{event_mix['any_numeric'].mean():.2%}")
print("Incidents with ANY redacted coords:", f"{event_mix['any_redacted'].mean():.2%}")
print("Incidents with MIXED numeric+redacted:", 
      f"{event_mix['mixed_numeric_and_redacted'].sum():,}", 
      f"({event_mix['mixed_numeric_and_redacted'].mean():.2%})")

# --- Did dedup pick REDACTED when raw had numeric anywhere? ---
dedup_flag = dedup[[id_col]].copy()
dedup_flag["dedup_is_redacted"] = redacted_mask(dedup).values

chk = (dedup_flag
       .merge(event_mix[["any_numeric"]], left_on=id_col, right_index=True, how="left", validate="1:1"))

problem = chk[(chk["any_numeric"] == True) & (chk["dedup_is_redacted"] == True)]

print("\n=== DEDUP SELECTION DIAGNOSTIC ===")
print("Incidents where RAW had numeric coords somewhere BUT DEDUP chose REDACTED:",
      f"{len(problem):,}", f"({len(problem)/len(dedup):.2%} of dedup incidents)")



=== BASIC SHAPES ===
Raw rows: 546,208 | Raw unique incidents: 324,386
Dedup rows: 324,386 | Dedup unique incidents: 324,386

=== REDACTED % (ROW-LEVEL) ===
Raw REDACTED %: 24.31%
Dedup REDACTED %: 22.86%

=== NUMERIC COORDS % (ROW-LEVEL) ===
Raw both-numeric %: 75.69%
Dedup both-numeric %: 77.14%

=== RAW INCIDENT-LEVEL COORD PATTERNS ===
Incidents with ANY numeric coords: 77.14%
Incidents with ANY redacted coords: 22.86%
Incidents with MIXED numeric+redacted: 0 (0.00%)

=== DEDUP SELECTION DIAGNOSTIC ===
Incidents where RAW had numeric coords somewhere BUT DEDUP chose REDACTED: 0 (0.00% of dedup incidents)


In [59]:
# --- 0) Pre-join baselines ---
n_before = len(df_call_map_joined)
n_events_before = df_call_map_joined["cad_event_number"].nunique(dropna=False) if "cad_event_number" in df_call_map_joined.columns else None

print("=== PRE-JOIN ===")
print(f"Rows (df_call_map_joined): {n_before:,}")
if n_events_before is not None:
    print(f"Unique cad_event_number: {n_events_before:,}  |  rows-per-event: {n_before/n_events_before:.3f}")

print("\ncare_tier distribution (rows):")
care_pre = (df_call_map_joined["care_tier"].astype(str).str.strip()
            .value_counts(dropna=False)
            .to_frame("count"))
care_pre["pct"] = (care_pre["count"] / care_pre["count"].sum() * 100).round(2)
print(care_pre)

# --- 1) Check right table uniqueness on join key (this is the usual blow-up cause) ---
print("\n=== RIGHT TABLE KEY CHECK (df_census) ===")
dup_keys = df_census["Call Data Neighbourhood"].astype(str).value_counts()
n_dup_keys = (dup_keys > 1).sum()
print(f"Distinct keys in df_map: {df_census['Call Data Neighbourhood'].nunique(dropna=False):,}")
print(f"Keys that appear >1 time (would cause blow-up): {n_dup_keys:,}")

if n_dup_keys > 0:
    print("\nTop duplicate join keys:")
    print(dup_keys[dup_keys > 1].head(20))

    # optional: inspect the actual duplicate rows
    # key_to_inspect = dup_keys[dup_keys > 1].index[0]
    # print(df_map[df_map["Call Data Neighbourhood"].astype(str) == str(key_to_inspect)]
    #       [["Call Data Neighbourhood", "Community Reporting Area Name"]])

=== PRE-JOIN ===
Rows (df_call_map_joined): 324,386
Unique cad_event_number: 324,386  |  rows-per-event: 1.000

care_tier distribution (rows):
                           count    pct
care_tier                              
Tier 0 - Traditional SPD  177837  54.82
Tier 1 - Potential CARE   125682  38.74
Tier 2 - Clearly CARE      20867   6.43

=== RIGHT TABLE KEY CHECK (df_census) ===
Distinct keys in df_map: 16
Keys that appear >1 time (would cause blow-up): 15

Top duplicate join keys:
Call Data Neighbourhood
FREMONT                     86
CAPITOL HILL                21
WALLINGFORD                 17
CENTRAL AREA/SQUIRE PARK    12
UNIVERSITY                   7
BALLARD SOUTH                5
BITTERLAKE                   4
FIRST HILL                   4
ALKI                         4
FAUNTLEROY SW                4
SOUTH PARK                   4
SOUTH DELRIDGE               3
BELLTOWN                     3
SODO                         2
nan                          2
Name: count, dtype: 

There is data blow up when joining on neighbourhood becauise multiple geoids map to one neighbourhood. to get geoid based on lat and long of call data we find ~23% of them are redacted. no particular neighbourhood stands out and there is no case where the redaction is in deduped file but not on raw file.

all code compute below this should be ignored. continue to #5.

In [62]:
import geopandas as gpd

# Your call data
df_call_map_joined_copy = df_call_map_joined.copy()

# Make points
g_calls = gpd.GeoDataFrame(
    df_call_map_joined_copy,
    geometry=gpd.points_from_xy(df_call_map_joined_copy["dispatch_longitude"], df_call_map_joined_copy["dispatch_latitude"]),
    crs="EPSG:4326"
)

# Load polygons (change path)
# For tracts:
poly_path = "../Data/tl_2024_53_tract/tl_2024_53_tract.shp"
# For block groups:
# poly_path = "../data_raw/geo/tl_2024_53_bg/tl_2024_53_bg.shp"

g_poly = gpd.read_file(poly_path).to_crs(g_calls.crs)

# OPTIONAL: restrict to King County to speed up (King County = 033)
# Works for both tracts and BGs because they contain COUNTYFP
if "COUNTYFP" in g_poly.columns:
    g_poly = g_poly[g_poly["COUNTYFP"] == "033"].copy()

# Spatial join: point within polygon
g_join = gpd.sjoin(
    g_calls[["cad_event_number", "geometry"]],
    g_poly[["GEOID", "geometry"]],
    how="left",
    predicate="within"
)

# Merge GEOID back to the call df (1 row per incident already)
df_calls_with_geoid = df_call_map_joined_copy.merge(
    g_join[["cad_event_number", "GEOID"]].drop_duplicates("cad_event_number"),
    on="cad_event_number",
    how="left",
    validate="1:1"
)

print("Missing GEOID fraction:", df_calls_with_geoid["GEOID"].isna().mean())
df_calls_with_geoid[["GEOID"]].head()


ValueError: could not convert string to float: 'REDACTED'

In [32]:
df_call_map_joined_prepared = df_call_map_joined.drop(columns=['_merge'])
df_call_rse_joined = df_call_map_joined.merge(
    df_census.rename(columns={'Community Reporting Area Name': 'Community Reporting Area Name rse', 'Call Data Neighbourhood': 'Call Data Neighbourhood rse'}),
    left_on='Community Reporting Area Name',
    right_on='Community Reporting Area Name rse',
    how='left',
    indicator='rse_merge_status'
)

checks:

In [33]:
# 1. Isolate the unmatched records
df_unmatched_rse_merge = df_call_rse_joined[df_call_rse_joined['rse_merge_status'] == 'left_only']

# 2. Identify the unique neighborhood names that failed to match
# We want to see the count to know how much data is being lost per name
unmatched_summary = (
    df_unmatched_rse_merge['dispatch_neighborhood']
    .value_counts(dropna=False)
    .rename_axis('unmatched_neighborhood')
    .reset_index(name='call_count')
)

print("--- Summary of Unmatched Neighborhoods ---")
if unmatched_summary.empty:
    print("Perfect match! No records were left behind.")
else:
    print(unmatched_summary)

--- Summary of Unmatched Neighborhoods ---
Perfect match! No records were left behind.


In [34]:
print_schema(df_call_rse_joined, "df_call_rse_joined")

Schema for df_call_rse_joined  |  rows=1,368,372 cols=98
  - cad_event_number: int64
  - cad_event_clearance_description: object
  - call_type: object
  - priority: int64
  - initial_call_type: object
  - final_call_type: object
  - cad_event_original_time_queued: datetime64[us]
  - cad_event_arrived_time: datetime64[us]
  - dispatch_precinct: object
  - dispatch_sector: object
  - dispatch_beat: object
  - dispatch_longitude: object
  - dispatch_latitude: object
  - dispatch_reporting_area: object
  - cad_event_response_category: object
  - call_sign_dispatch_id: object
  - call_sign_dispatch_time: datetime64[us]
  - first_care_call_sign_at_scene_time: datetime64[us]
  - first_care_call_sign_dispatch_time: datetime64[us]
  - first_co_response_call_sign_at_scene_time: datetime64[us]
  - first_co_response_call_sign_dispatch_time: datetime64[us]
  - first_spd_call_sign_at_scene_time: datetime64[us]
  - first_spd_call_sign_dispatch_time: datetime64[us]
  - last_care_call_sign_in_service_t

In [35]:

# ---- CONFIG ----
DF_NAME = "df_call_rse_joined"          # change if needed
OUT_BASENAME = "call_map_rse_joined"        # prefix for saved files
OUT_DIR = "../data_processed/"                           # or e.g. "./outputs"

# ---- GET DF ----
df = globals().get(DF_NAME)
if df is None or not isinstance(df, pd.DataFrame):
    raise ValueError(f"DataFrame '{DF_NAME}' not found. Set DF_NAME to your dataframe variable name.")

# ---- 1) SAVE SCHEMA AS CSV ----
schema_df = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(t) for t in df.dtypes],
    "n_null": df.isna().sum().values,
    "pct_null": (df.isna().mean() * 100).round(2).values,
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
})

schema_path = f"{OUT_DIR}/{OUT_BASENAME}_schema.csv"
schema_df.to_csv(schema_path, index=False)

# ---- 2) SAVE DF AS CSV ----
csv_path = f"{OUT_DIR}/{OUT_BASENAME}.csv"
df.to_csv(csv_path, index=False)

# ---- 3) SAVE DF AS PARQUET ----
parquet_path = f"{OUT_DIR}/{OUT_BASENAME}.parquet"
df.to_parquet(parquet_path, index=False)  # requires pyarrow or fastparquet installed

print("Saved:")
print("  schema:", schema_path)
print("  csv   :", csv_path)
print("  parquet:", parquet_path)


Saved:
  schema: ../data_processed//call_map_rse_joined_schema.csv
  csv   : ../data_processed//call_map_rse_joined.csv
  parquet: ../data_processed//call_map_rse_joined.parquet


csv ranamed to: call_map_rse_joined_blownupinjoin.csv. others deleted.

In [42]:
from pathlib import Path
import csv

top_row_df = df_call_rse_joined.head(3)
output_string = top_row_df.to_string(index=False)

file_path = Path("../data_processed/samples/df_call_rse_joined_sample.txt")
file_path.parent.mkdir(parents=True, exist_ok=True)  # <-- create folders
filename = "../data_processed/samples/df_call_rse_joined_sample.csv"
file_path.write_text(output_string)
print(f"Top row successfully written to {file_path}")


Top row successfully written to ../data_processed/samples/df_call_rse_joined_sample.txt


In [43]:
# Get unique values
unique_tiernames = df_call_rse_joined['care_tier'].unique()

print(unique_tiernames)

['Tier 0 - Traditional SPD' 'Tier 1 - Potential CARE'
 'Tier 2 - Clearly CARE']


In [44]:
df_call_rse_joined_tier2 = df_call_rse_joined[df_call_rse_joined['care_tier'] == 'Tier 2 - Clearly CARE']



In [45]:
# Filter (your line)
df_call_rse_joined_tier2 = df_call_rse_joined[df_call_rse_joined["care_tier"] == "Tier 2 - Clearly CARE"].copy()

# 1) Row counts + percent kept
n0 = len(df_call_rse_joined)
n1 = len(df_call_rse_joined_tier2)
print(f"Rows before: {n0:,}")
print(f"Rows Tier 2: {n1:,} ({n1/n0:.1%} of all rows)")

# 2) care_tier distribution (to confirm exact string match is doing what you expect)
print("\ncare_tier value_counts (top 15):")
print(df_call_rse_joined["care_tier"].value_counts(dropna=False).head(15))

# 3) Assert only Tier 2 remains (sanity)
print("\nUnique care_tier values in Tier2 df:")
print(df_call_rse_joined_tier2["care_tier"].value_counts(dropna=False))

# 4) Missingness quick scan (top 20 columns by null rate)
null_rate = df_call_rse_joined_tier2.isna().mean().sort_values(ascending=False)
print("\nTop null-rate columns in Tier2 df:")
print((null_rate.head(20) * 100).round(1).astype(str) + "%")

# 5) Duplicate check (edit id col name if you have one)
id_candidates = ["cad_event_number", "event_number", "incident_id", "cad_id"]
id_col = next((c for c in id_candidates if c in df_call_rse_joined_tier2.columns), None)

if id_col:
    n_dist = df_call_rse_joined_tier2[id_col].nunique(dropna=False)
    print(f"\nID column used: {id_col}")
    print(f"Distinct IDs: {n_dist:,} (rows per ID avg: {n1/n_dist:.2f})")
else:
    print("\nNo obvious ID column found among:", id_candidates)

# 6) RSE sanity (edit rse column name if needed)
rse_candidates = ["rse", "RSE", "rse_score", "rse_index"]
rse_col = next((c for c in rse_candidates if c in df_call_rse_joined_tier2.columns), None)

if rse_col:
    print(f"\nRSE column used: {rse_col}")
    print(df_call_rse_joined_tier2[rse_col].describe())
    print("RSE missing:", df_call_rse_joined_tier2[rse_col].isna().mean().round(3))
else:
    print("\nNo obvious RSE column found among:", rse_candidates)

# 7) Neighborhood coverage (edit to your geography col)
geo_candidates = [
    "Community Reporting Area Name", "community_reporting_area",
    "dispatch_neighborhood", "neighborhood", "census_tract"
]
geo_col = next((c for c in geo_candidates if c in df_call_rse_joined_tier2.columns), None)

if geo_col:
    print(f"\nGeography column used: {geo_col}")
    print("Distinct geos:", df_call_rse_joined_tier2[geo_col].nunique(dropna=False))
    print("Top 10 geos by Tier2 rows:")
    print(df_call_rse_joined_tier2[geo_col].value_counts(dropna=False).head(10))
    print("Geo missing:", df_call_rse_joined_tier2[geo_col].isna().mean().round(3))
else:
    print("\nNo obvious geography column found among:", geo_candidates)


Rows before: 1,368,372
Rows Tier 2: 92,292 (6.7% of all rows)

care_tier value_counts (top 15):
care_tier
Tier 0 - Traditional SPD    729686
Tier 1 - Potential CARE     546394
Tier 2 - Clearly CARE        92292
Name: count, dtype: int64

Unique care_tier values in Tier2 df:
care_tier
Tier 2 - Clearly CARE    92292
Name: count, dtype: int64

Top null-rate columns in Tier2 df:
first_co_response_call_sign_response_time_s_          94.9%
first_co_response_call_sign_at_scene_time             94.9%
first_co_response_call_sign_dispatch_delay_time_s_    94.9%
co_response_call_sign_total_service_time_s_           94.9%
last_co_response_call_sign_in_service_time            94.9%
first_co_response_call_sign_dispatch_time             94.9%
first_care_call_sign_response_time_s_                 82.7%
first_care_call_sign_at_scene_time                    82.7%
first_care_call_sign_dispatch_delay_time_s_           82.6%
care_call_sign_total_service_time_s_                  82.6%
last_care_call_sign_in

In [46]:
(df_call_rse_joined["care_tier"]
 .astype(str).str.strip()
 .value_counts(dropna=False)
 .to_frame("row_count")
 .assign(row_pct=lambda x: (x["row_count"] / x["row_count"].sum() * 100).round(2))
)


,row_count,row_pct
care_tier,,
Tier 0 - Traditional SPD,729686,53.33
Tier 1 - Potential CARE,546394,39.93
Tier 2 - Clearly CARE,92292,6.74


In [47]:
id_col = "cad_event_number"  # <- change if needed

tmp = df_call_rse_joined.copy()
tmp["care_tier_clean"] = tmp["care_tier"].astype(str).str.strip()

# event-level (unique cad_event_number) distribution
event_counts = (tmp.dropna(subset=[id_col])
                  .groupby("care_tier_clean")[id_col]
                  .nunique()
                  .sort_values(ascending=False)
                  .to_frame("n_events"))

event_counts["pct_events"] = (event_counts["n_events"] / event_counts["n_events"].sum() * 100).round(2)
event_counts


,n_events,pct_events
care_tier_clean,,
Tier 0 - Traditional SPD,177837,54.82
Tier 1 - Potential CARE,125682,38.74
Tier 2 - Clearly CARE,20867,6.43
